# Notebook 12 — Volumetric Mapping

**Vision & 3D Mapping Workshop** | Block 3: 3D Mapping

---

## Why This Matters

Individual depth frames give you a single snapshot of a scene, but robots and AR systems need
a **persistent, globally-consistent 3D map**. Volumetric mapping fuses many noisy depth
measurements into a single dense representation by discretising 3D space into a regular grid
of **voxels**.

Two dominant paradigms exist:

| Representation | Stores | Best for |
|----------------|--------|----------|
| **TSDF** (Truncated Signed Distance Function) | Distance to nearest surface | High-quality surface reconstruction |
| **Occupancy Grid** | Probability of being occupied | Motion planning & navigation |

This notebook derives both from first principles, implements them from scratch, and compares them
on the same synthetic data.

### What You'll Learn

1. **TSDF integration** — the Curless & Levoy (1996) algorithm, implemented voxel by voxel
2. **Marching Cubes** — extracting triangle meshes from implicit surfaces
3. **Probabilistic occupancy grids** — log-odds Bayesian updates with Bresenham ray casting
4. **When to use which** — TSDF vs. occupancy grids, trade-offs in practice
5. **Efficient storage** — OctoMap, voxel hashing, and why dense grids don't scale
6. **KinectFusion** — the system that brought real-time TSDF to the world

### Prerequisites
- Depth imaging and camera models (Notebooks 03, 08, 09)
- SE(3) rigid-body transforms (Notebook 04)
- Point cloud basics (Notebook 11)

### References
- Curless & Levoy, "A Volumetric Method for Building Complex Models from Range Images", SIGGRAPH 1996
- Lorensen & Cline, "Marching Cubes: A High Resolution 3D Surface Construction Algorithm", SIGGRAPH 1987
- Elfes, "Using Occupancy Grids for Mobile Robot Perception and Navigation", Computer 1989
- Hornung et al., "OctoMap: An Efficient Probabilistic 3D Mapping Framework", Autonomous Robots 2013
- Newcombe et al., "KinectFusion: Real-Time Dense Surface Mapping and Tracking", ISMAR 2011

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["image.cmap"] = "gray"
np.set_printoptions(precision=4, suppress=True)

from src.transforms import rotation_matrix_from_euler, se3_from_Rt
from src.camera import CameraIntrinsics
from src.pointcloud import depth_to_pointcloud
from src.tsdf import TSDFVolume
from src.occupancy import OccupancyGrid

---
## Synthetic Scene & Depth Rendering

Before diving into volumetric fusion, we need depth data. We'll build a simple synthetic scene
(a room with a box and a sphere) and render depth maps by projecting the known geometry into
the camera. This gives us perfect ground-truth depth from multiple viewpoints.

**Scene description:**
- A room: 6 planes forming a 2 m × 2 m × 2 m cube centred at the origin
- A sphere (radius 0.25 m) at position (-0.3, -0.2, 0.0)

We'll generate **12 depth frames** from cameras arranged on a circle looking inward.

In [ ]:
def render_depth_sphere(cam_pos, cam_target, K, img_h, img_w,
                        sphere_centre, sphere_radius):
    """Ray-cast a sphere and return per-pixel depth (0 = miss)."""
    forward = cam_target - cam_pos
    forward = forward / np.linalg.norm(forward)

    right = np.cross(forward, np.array([0.0, 0.0, 1.0]))
    norm_r = np.linalg.norm(right)
    if norm_r < 1e-6:
        right = np.cross(forward, np.array([0.0, 1.0, 0.0]))
        norm_r = np.linalg.norm(right)
    right /= norm_r
    up = np.cross(right, forward)
    up /= np.linalg.norm(up)

    R_cam = np.stack([right, -up, forward], axis=0)

    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]

    depth = np.zeros((img_h, img_w), dtype=np.float64)

    for v in range(img_h):
        for u in range(img_w):
            ray_cam = np.array([(u - cx) / fx, (v - cy) / fy, 1.0])
            ray_cam /= np.linalg.norm(ray_cam)
            ray_world = R_cam.T @ ray_cam

            oc = cam_pos - sphere_centre
            a = np.dot(ray_world, ray_world)
            b = 2.0 * np.dot(oc, ray_world)
            c = np.dot(oc, oc) - sphere_radius ** 2
            disc = b * b - 4 * a * c
            if disc >= 0:
                t = (-b - np.sqrt(disc)) / (2.0 * a)
                if t > 0.01:
                    hit = cam_pos + t * ray_world
                    p_cam = R_cam @ (hit - cam_pos)
                    depth[v, u] = p_cam[2]
    return depth, R_cam, cam_pos


def render_depth_planes(cam_pos, cam_target, K, img_h, img_w, planes):
    """Ray-cast a set of planes. Each plane: (normal, d) where n·x = d."""
    forward = cam_target - cam_pos
    forward = forward / np.linalg.norm(forward)

    right = np.cross(forward, np.array([0.0, 0.0, 1.0]))
    norm_r = np.linalg.norm(right)
    if norm_r < 1e-6:
        right = np.cross(forward, np.array([0.0, 1.0, 0.0]))
        norm_r = np.linalg.norm(right)
    right /= norm_r
    up = np.cross(right, forward)
    up /= np.linalg.norm(up)

    R_cam = np.stack([right, -up, forward], axis=0)

    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]

    depth = np.full((img_h, img_w), np.inf, dtype=np.float64)

    for v in range(img_h):
        for u in range(img_w):
            ray_cam = np.array([(u - cx) / fx, (v - cy) / fy, 1.0])
            ray_cam /= np.linalg.norm(ray_cam)
            ray_world = R_cam.T @ ray_cam

            for normal, d in planes:
                denom = np.dot(normal, ray_world)
                if abs(denom) < 1e-8:
                    continue
                t = (d - np.dot(normal, cam_pos)) / denom
                if t > 0.01:
                    hit = cam_pos + t * ray_world
                    if np.all(np.abs(hit) <= 1.01):
                        p_cam = R_cam @ (hit - cam_pos)
                        if p_cam[2] < depth[v, u]:
                            depth[v, u] = p_cam[2]

    depth[depth == np.inf] = 0.0
    return depth, R_cam, cam_pos


def render_scene(cam_pos, cam_target, K, img_h, img_w):
    """Render depth for the full synthetic scene (room + sphere)."""
    planes = [
        (np.array([ 1, 0, 0.]), 1.0),   # +x wall
        (np.array([-1, 0, 0.]), 1.0),   # -x wall
        (np.array([0,  1, 0.]), 1.0),   # +y wall
        (np.array([0, -1, 0.]), 1.0),   # -y wall
        (np.array([0, 0,  1.]), 1.0),   # ceiling
        (np.array([0, 0, -1.]), 1.0),   # floor
    ]

    depth_p, R_cam, _ = render_depth_planes(cam_pos, cam_target, K,
                                            img_h, img_w, planes)
    depth_s, _, _ = render_depth_sphere(cam_pos, cam_target, K,
                                        img_h, img_w,
                                        sphere_centre=np.array([-0.3, -0.2, 0.0]),
                                        sphere_radius=0.25)

    depth = depth_p.copy()
    mask = (depth_s > 0) & ((depth_s < depth) | (depth <= 0))
    depth[mask] = depth_s[mask]

    T_cam_to_world = np.eye(4)
    T_cam_to_world[:3, :3] = R_cam.T
    T_cam_to_world[:3, 3] = cam_pos

    return depth, T_cam_to_world

In [ ]:
IMG_H, IMG_W = 60, 80
K = np.array([
    [80.0, 0.0, 40.0],
    [0.0, 80.0, 30.0],
    [0.0,  0.0,  1.0]
])

n_frames = 12
radius = 0.8
cam_target = np.array([0.0, 0.0, 0.0])

depth_frames = []
poses = []

for i in range(n_frames):
    angle = 2.0 * np.pi * i / n_frames
    cam_pos = np.array([
        radius * np.cos(angle),
        radius * np.sin(angle),
        0.15 * np.sin(2 * angle)
    ])
    depth, T_cw = render_scene(cam_pos, cam_target, K, IMG_H, IMG_W)
    depth_frames.append(depth)
    poses.append(T_cw)

print(f"Generated {len(depth_frames)} depth frames, "
      f"resolution {IMG_H}x{IMG_W}")
print(f"Depth range (frame 0): [{depth_frames[0][depth_frames[0]>0].min():.2f}, "
      f"{depth_frames[0].max():.2f}] m")

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for i, ax in enumerate(axes.flat):
    d = depth_frames[i].copy()
    d[d <= 0] = np.nan
    im = ax.imshow(d, cmap="viridis")
    ax.set_title(f"Frame {i}")
    ax.axis("off")
fig.suptitle("Synthetic Depth Frames (12 views around scene)", fontsize=14)
plt.tight_layout()
plt.show()

---
## 1. TSDF Volume — Full Derivation (Curless & Levoy 1996)

### 1.1 The Core Idea

A **Truncated Signed Distance Function (TSDF)** represents surfaces implicitly. Instead of
storing triangles or points, we discretise 3D space into a regular grid of voxels, and at
each voxel we store the **signed distance** to the nearest observed surface:

$$\text{SDF}(\mathbf{v}) = \begin{cases} > 0 & \text{voxel is in front of the surface (free space)} \\ = 0 & \text{voxel is on the surface} \\ < 0 & \text{voxel is behind the surface (inside the object)} \end{cases}$$

The surface is the **zero-level set** $\{\mathbf{v} : \text{SDF}(\mathbf{v}) = 0\}$.

### 1.2 Voxel Data

Each voxel stores a pair $(\text{tsdf}, w)$:
- $\text{tsdf} \in [-1, 1]$: normalised truncated signed distance
- $w \geq 0$: accumulated weight (observation confidence)

### 1.3 Integration Algorithm

Given a depth image $D$ captured from a camera with intrinsics $K$ and pose $T_{\text{cam} \to \text{world}}$,
the update for each voxel $\mathbf{v} = (v_x, v_y, v_z)^\top$ proceeds as follows:

**Step 1: World to camera frame.** Transform the voxel to camera coordinates:

$$\mathbf{p}_{\text{cam}} = T_{\text{world} \to \text{cam}} \cdot \begin{pmatrix} v_x \\ v_y \\ v_z \\ 1 \end{pmatrix} = T_{\text{cam} \to \text{world}}^{-1} \cdot \tilde{\mathbf{v}}$$

where $T_{\text{world} \to \text{cam}} = T_{\text{cam} \to \text{world}}^{-1}$ and $\tilde{\mathbf{v}}$ is the
homogeneous representation of $\mathbf{v}$.

**Step 2: Project to pixel coordinates.** Using the pinhole camera model:

$$\begin{pmatrix} u \\ v \\ 1 \end{pmatrix} \sim K \begin{pmatrix} X_c \\ Y_c \\ Z_c \end{pmatrix} \implies u = f_x \frac{X_c}{Z_c} + c_x, \quad v = f_y \frac{Y_c}{Z_c} + c_y$$

Discard voxels that project outside the image or have $Z_c \leq 0$ (behind the camera).

**Step 3: Compute the raw signed distance.** Look up the observed depth at pixel $(u, v)$:

$$d_{\text{obs}} = D[\text{round}(v),\; \text{round}(u)]$$

The signed distance is the difference between the observed depth and the voxel's depth:

$$\text{sdf} = d_{\text{obs}} - Z_c$$

Interpretation:
- $\text{sdf} > 0$: the voxel is **in front** of the observed surface (closer to the camera)
- $\text{sdf} < 0$: the voxel is **behind** the observed surface
- $\text{sdf} = 0$: the voxel is exactly on the surface

**Step 4: Truncate.** Normalise by the truncation distance $\mu$ and clamp:

$$\text{tsdf}_{\text{new}} = \text{clamp}\!\left(\frac{\text{sdf}}{\mu},\; -1,\; 1\right)$$

The truncation distance $\mu$ limits the "reach" of each depth observation. Typically
$\mu \approx 3 \times \text{voxel\_size}$. Voxels with $\text{sdf} < -\mu$ are discarded
entirely (too far behind the surface to be informative).

**Step 5: Weighted running average.** Fuse the new measurement with the stored value:

$$\text{tsdf} = \frac{w_{\text{old}} \cdot \text{tsdf}_{\text{old}} + w_{\text{new}} \cdot \text{tsdf}_{\text{new}}}{w_{\text{old}} + w_{\text{new}}}$$

$$w = \min(w_{\text{old}} + w_{\text{new}},\; w_{\max})$$

The weight clamping prevents any single region from becoming immutable (allows adaptation
to changes). This weighted average is the key insight of Curless & Levoy — it elegantly
handles noise, multiple observations, and yields smooth surfaces.

**Observation-dependent weight.** The new-observation weight $w_{\text{new}}$ need not be
constant — it should reflect measurement confidence. For a voxel observed at depth $Z_c$
with surface normal making angle $\theta$ to the viewing ray:

$$w_{\text{new}} = \frac{\cos\theta}{Z_c^2}$$

**Why this formula:**

- $\cos\theta$ (angle of incidence): Measurements at grazing angles ($\theta \to 90°$)
  have larger depth uncertainty because a small angular error maps to a large depth
  displacement. At normal incidence ($\theta = 0$) the measurement is most reliable.
- $1/Z_c^2$: Depth noise grows with distance. For a structured-light or ToF sensor,
  depth variance scales approximately as $\sigma_z^2 \propto Z_c^2$, so the inverse-variance
  weight is $1/Z_c^2$. For stereo, the relationship is even steeper ($\propto Z_c^4$).

In our implementation we use $w_{\text{new}} = 1$ for simplicity (all observations weighted
equally), which is common in educational and real-time settings (KinectFusion uses $w = 1$).
Production systems like InfiniTAM and BundleFusion use depth- and angle-dependent weights
for improved surface quality.

### 1.4 Why Weighted Averaging Works

If each depth observation has i.i.d. Gaussian noise $\epsilon \sim \mathcal{N}(0, \sigma^2)$,
then after $n$ measurements with equal weight the fused TSDF has variance:

$$\text{Var}(\text{tsdf}_{\text{fused}}) = \frac{\sigma^2 / \mu^2}{n}$$

The noise decreases as $O(1/n)$ — more observations yield cleaner surfaces.

In [ ]:
class TSDFVolumeFromScratch:
    """TSDF volume implemented from scratch (Curless & Levoy 1996)."""

    def __init__(self, vol_bounds, voxel_size=0.04, trunc_dist=0.12):
        self.vol_bounds = np.asarray(vol_bounds, dtype=np.float64)
        self.voxel_size = voxel_size
        self.trunc_dist = trunc_dist

        self.origin = self.vol_bounds[:, 0].copy()
        self.dims = np.ceil(
            (self.vol_bounds[:, 1] - self.vol_bounds[:, 0]) / voxel_size
        ).astype(np.int32)

        self.tsdf_vol = np.ones(self.dims, dtype=np.float32)
        self.weight_vol = np.zeros(self.dims, dtype=np.float32)

        self._voxel_coords = self._build_voxel_coords()

    def _build_voxel_coords(self):
        xv = np.arange(self.dims[0])
        yv = np.arange(self.dims[1])
        zv = np.arange(self.dims[2])
        grid = np.stack(
            np.meshgrid(xv, yv, zv, indexing="ij"), axis=-1
        )
        coords = grid.reshape(-1, 3).astype(np.float64) * self.voxel_size
        coords += self.origin + self.voxel_size / 2.0
        return coords

    def integrate(self, depth, K, T_cam_to_world, weight=1.0):
        """Integrate a single depth frame into the volume."""
        h, w = depth.shape[:2]
        T_world_to_cam = np.linalg.inv(T_cam_to_world)

        n = self._voxel_coords.shape[0]
        ones = np.ones((n, 1), dtype=np.float64)
        hom = np.concatenate([self._voxel_coords, ones], axis=1)
        cam = (T_world_to_cam @ hom.T).T

        cam_z = cam[:, 2]

        fx, fy = K[0, 0], K[1, 1]
        cx, cy = K[0, 2], K[1, 2]
        pix_x = fx * (cam[:, 0] / cam_z) + cx
        pix_y = fy * (cam[:, 1] / cam_z) + cy

        valid = (
            (cam_z > 0)
            & (pix_x >= 0) & (pix_x < w - 1)
            & (pix_y >= 0) & (pix_y < h - 1)
        )

        pix_xi = np.round(pix_x).astype(np.int32)
        pix_yi = np.round(pix_y).astype(np.int32)
        pix_xi = np.clip(pix_xi, 0, w - 1)
        pix_yi = np.clip(pix_yi, 0, h - 1)

        d_obs = depth[pix_yi, pix_xi]
        valid &= d_obs > 0

        sdf = d_obs - cam_z
        valid &= sdf >= -self.trunc_dist

        tsdf_new = np.clip(sdf / self.trunc_dist, -1.0, 1.0)

        valid_idx = np.where(valid)[0]
        vi = np.unravel_index(valid_idx, self.dims)

        w_old = self.weight_vol[vi]
        tsdf_old = self.tsdf_vol[vi]
        w_new = weight

        w_sum = w_old + w_new
        self.tsdf_vol[vi] = (
            (w_old * tsdf_old + w_new * tsdf_new[valid_idx]) / w_sum
        )
        self.weight_vol[vi] = np.minimum(w_sum, 255.0)

    def get_tsdf_slice(self, axis, index):
        """Return a 2D slice of the TSDF volume along the given axis."""
        if axis == 0:
            return self.tsdf_vol[index, :, :]
        elif axis == 1:
            return self.tsdf_vol[:, index, :]
        else:
            return self.tsdf_vol[:, :, index]


print("TSDFVolumeFromScratch class defined.")

In [ ]:
vol_bounds = np.array([
    [-1.0, 1.0],
    [-1.0, 1.0],
    [-1.0, 1.0]
])
voxel_size = 0.04
trunc_dist = 3 * voxel_size

tsdf = TSDFVolumeFromScratch(vol_bounds, voxel_size, trunc_dist)
print(f"TSDF volume dimensions: {tsdf.dims}")
print(f"Total voxels: {np.prod(tsdf.dims):,}")
print(f"Voxel size: {voxel_size} m")
print(f"Truncation distance: {trunc_dist:.3f} m")

In [ ]:
for i in range(n_frames):
    tsdf.integrate(depth_frames[i], K, poses[i])
    observed = np.sum(tsdf.weight_vol > 0)
    print(f"  Frame {i:2d}: {observed:6d} observed voxels "
          f"({100*observed/np.prod(tsdf.dims):.1f}%)")

print(f"\nFinal TSDF range: [{tsdf.tsdf_vol[tsdf.weight_vol>0].min():.3f}, "
      f"{tsdf.tsdf_vol[tsdf.weight_vol>0].max():.3f}]")
print(f"Max weight: {tsdf.weight_vol.max():.0f}")

### 1.4 Visualising the Fused TSDF

Let's examine cross-sectional slices of the fused TSDF volume.  The zero-crossing
(white band between red and blue) marks the reconstructed surface.

In [ ]:
mid_z = tsdf.dims[2] // 2
mid_y = tsdf.dims[1] // 2
mid_x = tsdf.dims[0] // 2

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

slice_xy = tsdf.get_tsdf_slice(2, mid_z).copy()
slice_xy[tsdf.weight_vol[:, :, mid_z] == 0] = np.nan
im0 = axes[0].imshow(slice_xy.T, origin="lower", cmap="RdBu",
                      vmin=-1, vmax=1)
axes[0].set_title(f"XY slice (z = {mid_z})")
axes[0].set_xlabel("X voxel")
axes[0].set_ylabel("Y voxel")
plt.colorbar(im0, ax=axes[0], label="TSDF")

slice_xz = tsdf.get_tsdf_slice(1, mid_y).copy()
slice_xz[tsdf.weight_vol[:, mid_y, :] == 0] = np.nan
im1 = axes[1].imshow(slice_xz.T, origin="lower", cmap="RdBu",
                      vmin=-1, vmax=1)
axes[1].set_title(f"XZ slice (y = {mid_y})")
axes[1].set_xlabel("X voxel")
axes[1].set_ylabel("Z voxel")
plt.colorbar(im1, ax=axes[1], label="TSDF")

slice_yz = tsdf.get_tsdf_slice(0, mid_x).copy()
slice_yz[tsdf.weight_vol[mid_x, :, :] == 0] = np.nan
im2 = axes[2].imshow(slice_yz.T, origin="lower", cmap="RdBu",
                      vmin=-1, vmax=1)
axes[2].set_title(f"YZ slice (x = {mid_x})")
axes[2].set_xlabel("Y voxel")
axes[2].set_ylabel("Z voxel")
plt.colorbar(im2, ax=axes[2], label="TSDF")

fig.suptitle("TSDF Volume Slices\n"
             "Red = behind surface (negative), Blue = free space (positive), "
             "White = zero-crossing (surface)",
             fontsize=13)
plt.tight_layout()
plt.show()

---
## 2. Marching Cubes — Mesh Extraction from TSDF

### 2.1 The Algorithm (Lorensen & Cline 1987)

Given an implicit surface defined by $f(\mathbf{x}) = 0$ (our TSDF zero-crossing), Marching
Cubes extracts a triangle mesh by examining each $2 \times 2 \times 2$ block of voxels:

**Step 1: Classify corners.** Each of the 8 corners of a voxel cube is classified as:
- **Inside** ($\text{tsdf} < 0$): behind the surface
- **Outside** ($\text{tsdf} \geq 0$): in front of the surface

With 8 corners and 2 possible states each, there are $2^8 = 256$ possible configurations.
By symmetry, these reduce to just **15 unique topological cases**:

**Symmetry reduction (256 → 15).** Two symmetry operations compress the case space:

1. **Complementarity:** Swapping inside ↔ outside (flipping all 8 bits) produces the same
   surface topology with reversed normals. This identifies each configuration with its
   complement, halving the count: $256 / 2 = 128$ independent cases (plus the all-same
   case that is self-complementary).

2. **Rotational symmetry:** The cube has 24 orientation-preserving rotational symmetries
   (the chiral octahedral group $O$ — the group of 24 rotations that map a cube to itself, excluding reflections). Configurations related by a rotation of the cube
   share the same surface topology. Applying this group action to the 128+1
   non-complementary cases yields exactly **15 equivalence classes**.

The 15 cases span a range of complexities:

| Case | Inside corners | Triangles | Description |
|:----:|:--------------:|:---------:|:---|
| 0 | 0 (or 8) | 0 | No surface (entirely free or occupied) |
| 1 | 1 | 1 | Single corner clipped → 1 triangle |
| 2 | 2 (adjacent) | 2 | Edge straddled → 2 triangles |
| 3 | 2 (diagonal) | 2 | Face-diagonal pair |
| 4 | 3 | 3 | L-shaped triple |
| … | … | … | … |
| 14 | 4 (tetrahedral) | 4 | Most complex base case |

**Ambiguous cases.** Cases 3, 6, 7, 10, 12, and 13 contain **face or body ambiguities**:
the corner signs alone do not uniquely determine whether the surface separates or connects
through a face/body diagonal. The original Lorensen–Cline algorithm resolves these
arbitrarily, which can produce holes. **Marching Cubes 33** (Chernyaev 1995) examines
face saddle points to resolve all ambiguities, expanding the table from 15 to 33 base cases.

**Step 2: Look up the triangulation.** A precomputed table maps each of the 256
configurations to a set of triangles that approximate the surface passing through
the cube. Each triangle's vertices lie on **edges** of the cube where the sign of the
TSDF changes.

**Step 3: Interpolate vertex positions.** For each edge with a sign change (one endpoint
inside, one outside), the exact surface crossing is estimated by linear interpolation:

$$\mathbf{p} = \mathbf{p}_1 + \frac{0 - \text{tsdf}_1}{\text{tsdf}_2 - \text{tsdf}_1} \cdot (\mathbf{p}_2 - \mathbf{p}_1)$$

where $\mathbf{p}_1$ and $\mathbf{p}_2$ are the two corner positions with TSDF values
$\text{tsdf}_1$ and $\text{tsdf}_2$ respectively. This places the vertex at the
zero-crossing along the edge.

### 2.2 The 256 Cases

The case index is computed as a binary number:

$$\text{case\_index} = \sum_{i=0}^{7} \begin{cases} 2^i & \text{if corner } i \text{ is inside} \\ 0 & \text{otherwise} \end{cases}$$

Each case maps to 0–5 triangles (0–15 edges). The lookup table has $256 \times 16$
entries (each row lists up to 5 triangles × 3 edge indices, terminated by $-1$).

### 2.3 Using scikit-image

The `skimage.measure.marching_cubes` function implements the full algorithm with the
standard lookup tables. It takes a 3D volume and an iso-level and returns:
- `verts`: $(V, 3)$ vertex positions (in voxel units)
- `faces`: $(F, 3)$ triangle indices
- `normals`: $(V, 3)$ per-vertex normals (estimated from gradient)
- `values`: $(V,)$ interpolated values at each vertex

In [ ]:
from skimage.measure import marching_cubes

tsdf_for_mc = tsdf.tsdf_vol.copy()
tsdf_for_mc[tsdf.weight_vol == 0] = 1.0

try:
    verts, faces, normals, _ = marching_cubes(tsdf_for_mc, level=0.0)
    verts_world = verts * tsdf.voxel_size + tsdf.origin

    print(f"Extracted mesh:")
    print(f"  Vertices: {verts_world.shape[0]:,}")
    print(f"  Triangles: {faces.shape[0]:,}")
    print(f"  Vertex range:")
    for ax_name, ax_i in zip("XYZ", range(3)):
        print(f"    {ax_name}: [{verts_world[:, ax_i].min():.3f}, "
              f"{verts_world[:, ax_i].max():.3f}]")
except (ValueError, RuntimeError) as e:
    print(f"Marching cubes found no surface: {e}")
    verts_world = np.zeros((0, 3))
    faces = np.zeros((0, 3), dtype=int)

In [ ]:
if len(verts_world) > 0:
    fig = plt.figure(figsize=(14, 6))

    ax1 = fig.add_subplot(121, projection="3d")
    stride = max(1, len(verts_world) // 5000)
    ax1.scatter(verts_world[::stride, 0],
               verts_world[::stride, 1],
               verts_world[::stride, 2],
               c=verts_world[::stride, 2], cmap="viridis",
               s=1, alpha=0.5)
    ax1.set_xlabel("X"); ax1.set_ylabel("Y"); ax1.set_zlabel("Z")
    ax1.set_title("Extracted Mesh Vertices")

    ax2 = fig.add_subplot(122, projection="3d")
    max_tris = 2000
    face_subset = faces[:max_tris]
    mesh_triangles = verts_world[face_subset]
    poly = Poly3DCollection(mesh_triangles, alpha=0.3,
                            facecolor="steelblue", edgecolor="gray",
                            linewidth=0.1)
    ax2.add_collection3d(poly)
    ax2.set_xlim(verts_world[:, 0].min(), verts_world[:, 0].max())
    ax2.set_ylim(verts_world[:, 1].min(), verts_world[:, 1].max())
    ax2.set_zlim(verts_world[:, 2].min(), verts_world[:, 2].max())
    ax2.set_xlabel("X"); ax2.set_ylabel("Y"); ax2.set_zlabel("Z")
    ax2.set_title(f"Mesh Surface (first {max_tris} triangles)")

    plt.suptitle("Marching Cubes Mesh Extraction from TSDF", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No mesh to visualize.")

### 2.4 Edge Interpolation — Worked Example

Consider two adjacent voxels $A$ and $B$ with:
- $\text{tsdf}_A = +0.6$ (outside the surface)
- $\text{tsdf}_B = -0.3$ (inside the surface)

The zero-crossing lies at:

$$t = \frac{0 - \text{tsdf}_A}{\text{tsdf}_B - \text{tsdf}_A} = \frac{0 - 0.6}{-0.3 - 0.6} = \frac{-0.6}{-0.9} = \frac{2}{3}$$

So the surface vertex is located $2/3$ of the way from $A$ to $B$:

$$\mathbf{p}_{\text{surface}} = \mathbf{p}_A + \frac{2}{3}(\mathbf{p}_B - \mathbf{p}_A)$$

This linear interpolation is exact when the SDF varies linearly between adjacent voxels
(a good approximation near the surface with small voxel sizes).

In [ ]:
tsdf_A, tsdf_B = 0.6, -0.3
pA, pB = np.array([1.0, 0.0, 0.0]), np.array([1.0, 0.04, 0.0])

t_interp = (0 - tsdf_A) / (tsdf_B - tsdf_A)
p_surface = pA + t_interp * (pB - pA)

print(f"Interpolation parameter t = {t_interp:.4f}")
print(f"Surface vertex at: {p_surface}")

fig, ax = plt.subplots(figsize=(8, 3))
x_vals = np.linspace(0, 1, 100)
tsdf_vals = tsdf_A + (tsdf_B - tsdf_A) * x_vals
ax.plot(x_vals, tsdf_vals, "b-", linewidth=2)
ax.axhline(0, color="k", linestyle="--", alpha=0.5)
ax.plot(0, tsdf_A, "ro", markersize=10, label=f"Voxel A (tsdf={tsdf_A})")
ax.plot(1, tsdf_B, "bs", markersize=10, label=f"Voxel B (tsdf={tsdf_B})")
ax.plot(t_interp, 0, "g*", markersize=15, label=f"Surface (t={t_interp:.3f})")
ax.set_xlabel("Normalised edge position")
ax.set_ylabel("TSDF value")
ax.set_title("Edge Interpolation for Marching Cubes")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Probabilistic Occupancy Grid — Log-Odds

### 3.1 Motivation

While TSDF excels at surface reconstruction, many robotics tasks (path planning, collision
avoidance) need to know **which regions of space are free, occupied, or unknown**. An
occupancy grid answers this question probabilistically.

### 3.2 Occupancy Probability

Each voxel $\mathbf{x}$ stores $p(\mathbf{x})$, the probability that it is occupied. Given
a sequence of measurements $z_{1:t}$, we want the posterior:

$$p(\mathbf{x} \mid z_{1:t})$$

Applying Bayes' rule recursively:

$$p(\mathbf{x} \mid z_{1:t}) = \frac{p(z_t \mid \mathbf{x}) \, p(\mathbf{x} \mid z_{1:t-1})}{p(z_t \mid z_{1:t-1})}$$

### 3.3 Log-Odds Representation

Working directly with probabilities is numerically unstable near 0 and 1, and requires
computing the normalisation constant. The **log-odds** representation avoids both issues.

Define the log-odds (logit) of the occupancy probability:

$$l(\mathbf{x}) = \log \frac{p(\mathbf{x})}{1 - p(\mathbf{x})}$$

Properties:
- $p = 0.5 \implies l = 0$ (unknown/prior)
- $p > 0.5 \implies l > 0$ (likely occupied)
- $p < 0.5 \implies l < 0$ (likely free)
- $p \to 1 \implies l \to +\infty$
- $p \to 0 \implies l \to -\infty$

The inverse mapping (log-odds → probability) is the **sigmoid** function:

$$p(\mathbf{x}) = \frac{1}{1 + e^{-l(\mathbf{x})}} = \text{sigmoid}(l(\mathbf{x}))$$

### 3.4 Bayes Update in Log-Odds

The Bayes update becomes a simple **addition** in log-odds:

$$l(\mathbf{x} \mid z_{1:t}) = l(\mathbf{x} \mid z_{1:t-1}) + l(\mathbf{x} \mid z_t) - l_0$$

where:
- $l(\mathbf{x} \mid z_t) = \log \frac{p(\mathbf{x} \mid z_t)}{1 - p(\mathbf{x} \mid z_t)}$ is the **inverse sensor model**
- $l_0 = \log \frac{p_0}{1 - p_0}$ is the log-odds of the prior (typically $p_0 = 0.5 \implies l_0 = 0$)

**Derivation.** Start with Bayes' rule for the odds ratio:

$$\frac{p(\mathbf{x} \mid z_{1:t})}{1 - p(\mathbf{x} \mid z_{1:t})} = \frac{p(z_t \mid \mathbf{x})}{p(z_t \mid \neg \mathbf{x})} \cdot \frac{p(\mathbf{x} \mid z_{1:t-1})}{1 - p(\mathbf{x} \mid z_{1:t-1})}$$

Taking logarithms:

$$l(\mathbf{x} \mid z_{1:t}) = \underbrace{\log \frac{p(z_t \mid \mathbf{x})}{p(z_t \mid \neg \mathbf{x})}}_{\text{log-likelihood ratio}} + l(\mathbf{x} \mid z_{1:t-1})$$

Rewriting the log-likelihood ratio using Bayes' rule on the inverse sensor model:

$$\log \frac{p(z_t \mid \mathbf{x})}{p(z_t \mid \neg \mathbf{x})} = l(\mathbf{x} \mid z_t) - l_0$$

This gives us the update rule $l_{\text{new}} = l_{\text{old}} + l_{\text{measurement}} - l_0$.

### 3.5 Forward Beam Model — Where the Log-Odds Values Come From

The update rule derived in §3.4 uses the **log-likelihood ratio** $\log\frac{p(z_t \mid m)}{p(z_t \mid \neg m)}$, where $m$ denotes "the cell is occupied". To assign concrete values we need the **forward (beam) sensor model** — the probability of observing a range measurement $z$ given what we know about the cell.

**Range sensor model.** Consider a single ray from the sensor origin through voxel $\mathbf{x}$ at distance $r$ from the sensor. The depth measurement $z$ is subject to Gaussian noise $\epsilon \sim \mathcal{N}(0, \sigma_z^2)$:

$$
p(z \mid m, r) = \begin{cases}
\displaystyle \frac{1}{\sqrt{2\pi}\,\sigma_z}\exp\!\left(-\frac{(z - r)^2}{2\sigma_z^2}\right) + p_{\text{rand}} & \text{cell occupied (sensor hits here)} \\[6pt]
p_{\text{rand}} & \text{cell free (sensor passes through)}
\end{cases}
$$

where $p_{\text{rand}}$ is a small uniform probability accounting for random noise and max-range returns. The complementary probability for the cell being free is modelled as uniform because the measurement provides no information about empty space at distance $r$.

**From beam model to log-odds.** The log-likelihood ratio for a cell **at the measured surface** ($r \approx z$) is:

$$
\log\frac{p(z \mid m)}{p(z \mid \neg m)} = \log\frac{\frac{1}{\sqrt{2\pi}\,\sigma_z} + p_{\text{rand}}}{p_{\text{rand}}} > 0
$$

This is a positive constant (given fixed $\sigma_z$ and $p_{\text{rand}}$) — the cell is more likely occupied. For a cell **between the sensor and the surface** ($r < z$), the measurement passed through it, so:

$$
\log\frac{p(z \mid m)}{p(z \mid \neg m)} = \log\frac{p_{\text{rand}}}{p_{\text{pass}}} < 0
$$

where $p_{\text{pass}} > p_{\text{rand}}$ because the measurement is consistent with the cell being free. This yields a negative constant.

In practice, the exact beam model parameters are absorbed into two tunable constants — the **inverse sensor model** log-odds values:

### 3.6 Inverse Sensor Model

For a depth sensor measuring distance $d$ at pixel $(u, v)$, the inverse sensor model assigns
log-odds updates to voxels along the ray:

$$l(\mathbf{x} \mid z_t) = \begin{cases}
l_{\text{free}} < 0 & \text{voxel is between camera and surface (free space)} \\
l_{\text{occ}} > 0 & \text{voxel is at the surface (occupied)} \\
l_0 = 0 & \text{voxel is beyond the surface (unknown)}
\end{cases}$$

Typical values: $l_{\text{free}} = -0.4$ (corresponding to $p \approx 0.40$), $l_{\text{occ}} = +0.85$ (corresponding to $p \approx 0.70$). These are conservative single-observation updates — the Bayesian accumulation over many observations drives probabilities toward 0 or 1 over time.

### 3.7 Clamping to Prevent Overconfidence

Without bounds, log-odds can grow arbitrarily large, making the grid unable to adapt to
changes. We clamp:

$$l(\mathbf{x}) \in [l_{\min},\; l_{\max}]$$

Typical: $l_{\max} = 3.5 \implies p_{\max} \approx 0.97$.

### 3.8 Bresenham 3D Line Algorithm

To determine which voxels lie along a ray, we use the 3D generalisation of Bresenham's
line algorithm. Given start $(x_0, y_0, z_0)$ and end $(x_1, y_1, z_1)$ in voxel coordinates:

1. Identify the **dominant axis** (largest $|\Delta|$)
2. Step along the dominant axis one voxel at a time
3. Track two error accumulators for the secondary axes
4. When an error exceeds the threshold, step in that secondary direction

The algorithm visits every voxel on the discrete line with $O(\max(|\Delta x|, |\Delta y|, |\Delta z|))$ operations
and no floating-point arithmetic (integer-only).

### 3.9 DDA (Digital Differential Analyser) Raycasting

An alternative to Bresenham is the **DDA** algorithm, which uses floating-point arithmetic
but generalises more naturally to non-axis-aligned rays in continuous space.

**Setup.** Given a ray origin $\mathbf{o}$ and direction $\mathbf{d}$ (unit vector), the
ray equation is $\mathbf{r}(t) = \mathbf{o} + t\,\mathbf{d}$.  We want to step through
voxels in order of increasing $t$.

**Key idea.** Compute, for each axis $a \in \{x, y, z\}$, the parametric step
$\Delta t_a$ to cross one voxel boundary along that axis:

$$
\Delta t_a = \frac{s}{|d_a|}
$$

where $s$ is the voxel size, and the initial distance $t_a^{(0)}$ from the ray origin to
the first boundary crossing along axis $a$:

$$
t_a^{(0)} = \begin{cases}
\displaystyle\frac{\bigl(\lfloor o_a / s \rfloor + 1\bigr) s - o_a}{d_a} & \text{if } d_a > 0 \\[8pt]
\displaystyle\frac{\lfloor o_a / s \rfloor \cdot s - o_a}{d_a} & \text{if } d_a < 0
\end{cases}
$$

**Traversal loop.** At each step, advance along the axis $a^*$ with the smallest
next-crossing parameter $t_a$:

$$
a^* = \arg\min_{a \in \{x,y,z\}} t_a, \qquad
\text{voxel}_{a^*} \mathrel{+}= \text{sign}(d_{a^*}), \qquad
t_{a^*} \mathrel{+}= \Delta t_{a^*}
$$

This visits exactly the voxels pierced by the ray, in front-to-back order, with
$O(1)$ work per voxel (one comparison + one addition). DDA is often preferred on
GPUs because it avoids the integer error-accumulator bookkeeping of Bresenham
and handles arbitrary ray directions without special-casing the dominant axis.

In [ ]:
def bresenham_3d(start, end):
    """3D Bresenham line algorithm — returns list of (x, y, z) voxel indices."""
    x0, y0, z0 = int(start[0]), int(start[1]), int(start[2])
    x1, y1, z1 = int(end[0]), int(end[1]), int(end[2])

    dx, dy, dz = abs(x1 - x0), abs(y1 - y0), abs(z1 - z0)
    sx = 1 if x1 > x0 else -1
    sy = 1 if y1 > y0 else -1
    sz = 1 if z1 > z0 else -1

    points = []

    if dx >= dy and dx >= dz:
        ey = 2 * dy - dx
        ez = 2 * dz - dx
        for _ in range(dx + 1):
            points.append((x0, y0, z0))
            if ey >= 0:
                y0 += sy; ey -= 2 * dx
            if ez >= 0:
                z0 += sz; ez -= 2 * dx
            ey += 2 * dy; ez += 2 * dz; x0 += sx

    elif dy >= dx and dy >= dz:
        ex = 2 * dx - dy
        ez = 2 * dz - dy
        for _ in range(dy + 1):
            points.append((x0, y0, z0))
            if ex >= 0:
                x0 += sx; ex -= 2 * dy
            if ez >= 0:
                z0 += sz; ez -= 2 * dy
            ex += 2 * dx; ez += 2 * dz; y0 += sy

    else:
        ex = 2 * dx - dz
        ey = 2 * dy - dz
        for _ in range(dz + 1):
            points.append((x0, y0, z0))
            if ex >= 0:
                x0 += sx; ex -= 2 * dz
            if ey >= 0:
                y0 += sy; ey -= 2 * dz
            ex += 2 * dx; ey += 2 * dy; z0 += sz

    if not points:
        points.append((int(start[0]), int(start[1]), int(start[2])))
    return points


ray = bresenham_3d((0, 0, 0), (7, 3, 2))
print(f"Bresenham ray from (0,0,0) to (7,3,2): {len(ray)} voxels")
print(f"  Path: {ray}")

In [ ]:
class OccupancyGridFromScratch:
    """Probabilistic 3D occupancy grid using log-odds representation."""

    def __init__(self, bounds, resolution=0.1,
                 log_odds_free=-0.4, log_odds_occ=0.85,
                 log_odds_max=3.5):
        self.bounds = np.asarray(bounds, dtype=np.float64)
        self.resolution = resolution
        self.log_odds_free = log_odds_free
        self.log_odds_occ = log_odds_occ
        self.log_odds_max = log_odds_max

        self.origin = self.bounds[:, 0].copy()
        dims = self.bounds[:, 1] - self.bounds[:, 0]
        self.grid_size = np.ceil(dims / resolution).astype(int)
        self.grid = np.zeros(tuple(self.grid_size), dtype=np.float64)

    def world_to_voxel(self, point):
        idx = ((point - self.origin) / self.resolution).astype(int)
        return int(idx[0]), int(idx[1]), int(idx[2])

    def in_bounds(self, i, j, k):
        return (0 <= i < self.grid_size[0]
                and 0 <= j < self.grid_size[1]
                and 0 <= k < self.grid_size[2])

    def update(self, depth, K, T_cam_to_world):
        """Update the occupancy grid with a depth observation."""
        H, W = depth.shape
        R = T_cam_to_world[:3, :3]
        t = T_cam_to_world[:3, 3]

        cam_origin_voxel = self.world_to_voxel(t)
        K_inv = np.linalg.inv(K)

        step = max(1, min(H, W) // 40)
        for v in range(0, H, step):
            for u in range(0, W, step):
                d = depth[v, u]
                if d <= 0.0:
                    continue

                pixel_h = np.array([u, v, 1.0])
                p_cam = d * (K_inv @ pixel_h)
                p_world = R @ p_cam + t

                end_voxel = self.world_to_voxel(p_world)
                if not self.in_bounds(*end_voxel):
                    continue

                ray_voxels = bresenham_3d(cam_origin_voxel, end_voxel)

                for vx, vy, vz in ray_voxels[:-1]:
                    if self.in_bounds(vx, vy, vz):
                        self.grid[vx, vy, vz] += self.log_odds_free

                ex, ey, ez = ray_voxels[-1]
                if self.in_bounds(ex, ey, ez):
                    self.grid[ex, ey, ez] += self.log_odds_occ

        np.clip(self.grid, -self.log_odds_max, self.log_odds_max,
                out=self.grid)

    def to_probability(self):
        return 1.0 / (1.0 + np.exp(-self.grid))

    def get_occupied_points(self, threshold=0.5):
        log_thresh = np.log(threshold / (1.0 - threshold))
        ix, iy, iz = np.where(self.grid > log_thresh)
        coords = np.stack([ix, iy, iz], axis=-1).astype(np.float64)
        return self.origin + (coords + 0.5) * self.resolution

    def get_free_points(self, threshold=0.3):
        log_thresh = np.log(threshold / (1.0 - threshold))
        ix, iy, iz = np.where(self.grid < log_thresh)
        coords = np.stack([ix, iy, iz], axis=-1).astype(np.float64)
        return self.origin + (coords + 0.5) * self.resolution


print("OccupancyGridFromScratch class defined.")

In [ ]:
occ_resolution = 0.08
occ_grid = OccupancyGridFromScratch(
    bounds=vol_bounds,
    resolution=occ_resolution,
    log_odds_free=-0.4,
    log_odds_occ=0.85,
    log_odds_max=3.5
)

print(f"Occupancy grid dimensions: {occ_grid.grid_size}")
print(f"Total voxels: {np.prod(occ_grid.grid_size):,}")
print(f"Resolution: {occ_resolution} m")

for i in range(n_frames):
    occ_grid.update(depth_frames[i], K, poses[i])
    prob = occ_grid.to_probability()
    n_occ = np.sum(prob > 0.6)
    n_free = np.sum(prob < 0.4)
    print(f"  Frame {i:2d}: {n_occ:5d} occupied, {n_free:5d} free")

In [ ]:
occ_pts = occ_grid.get_occupied_points(threshold=0.6)
free_pts = occ_grid.get_free_points(threshold=0.3)

print(f"Occupied voxels: {len(occ_pts):,}")
print(f"Free voxels: {len(free_pts):,}")

fig = plt.figure(figsize=(16, 6))

ax1 = fig.add_subplot(121, projection="3d")
if len(occ_pts) > 0:
    stride = max(1, len(occ_pts) // 3000)
    ax1.scatter(occ_pts[::stride, 0], occ_pts[::stride, 1],
               occ_pts[::stride, 2], c="red", s=2, alpha=0.6,
               label="Occupied")
ax1.set_xlabel("X"); ax1.set_ylabel("Y"); ax1.set_zlabel("Z")
ax1.set_title("Occupied Voxels (p > 0.6)")
ax1.legend()

ax2 = fig.add_subplot(122, projection="3d")
if len(free_pts) > 0:
    stride = max(1, len(free_pts) // 3000)
    ax2.scatter(free_pts[::stride, 0], free_pts[::stride, 1],
               free_pts[::stride, 2], c="skyblue", s=1, alpha=0.2,
               label="Free")
if len(occ_pts) > 0:
    stride = max(1, len(occ_pts) // 2000)
    ax2.scatter(occ_pts[::stride, 0], occ_pts[::stride, 1],
               occ_pts[::stride, 2], c="red", s=3, alpha=0.8,
               label="Occupied")
ax2.set_xlabel("X"); ax2.set_ylabel("Y"); ax2.set_zlabel("Z")
ax2.set_title("Occupied + Free Space")
ax2.legend()

plt.suptitle("Probabilistic Occupancy Grid", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
prob_grid = occ_grid.to_probability()
mid_z = occ_grid.grid_size[2] // 2

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im0 = axes[0].imshow(prob_grid[:, :, mid_z].T, origin="lower",
                      cmap="RdYlGn_r", vmin=0, vmax=1)
axes[0].set_title(f"Occupancy Probability — XY (z={mid_z})")
axes[0].set_xlabel("X voxel"); axes[0].set_ylabel("Y voxel")
plt.colorbar(im0, ax=axes[0], label="p(occupied)")

mid_y = occ_grid.grid_size[1] // 2
im1 = axes[1].imshow(prob_grid[:, mid_y, :].T, origin="lower",
                      cmap="RdYlGn_r", vmin=0, vmax=1)
axes[1].set_title(f"Occupancy Probability — XZ (y={mid_y})")
axes[1].set_xlabel("X voxel"); axes[1].set_ylabel("Z voxel")
plt.colorbar(im1, ax=axes[1], label="p(occupied)")

im2 = axes[2].imshow(occ_grid.grid[:, :, mid_z].T, origin="lower",
                      cmap="RdBu_r",
                      vmin=-occ_grid.log_odds_max,
                      vmax=occ_grid.log_odds_max)
axes[2].set_title(f"Log-Odds Values — XY (z={mid_z})")
axes[2].set_xlabel("X voxel"); axes[2].set_ylabel("Y voxel")
plt.colorbar(im2, ax=axes[2], label="Log-odds")

plt.suptitle("Occupancy Grid Slices", fontsize=14)
plt.tight_layout()
plt.show()

---
## 4. Comparison — TSDF vs. Occupancy Grid

### 4.1 Conceptual Differences

| Property | TSDF | Occupancy Grid |
|----------|------|----------------|
| **Stores** | Signed distance to surface | Probability of being occupied |
| **Surface model** | Implicit (zero-level set) | None (binary classification) |
| **Free space** | Positive values (implicit) | Explicit (low probability) |
| **Mesh extraction** | Marching Cubes on zero-crossing | Not directly applicable |
| **Update rule** | Weighted running average | Bayesian (log-odds addition) |
| **Ray casting** | Not required (projective) | Required (Bresenham/DDA) |
| **Surface quality** | High — sub-voxel accuracy via interpolation | Low — voxel-level resolution |
| **Best for** | 3D reconstruction, rendering | Path planning, collision checking |

### 4.2 Key Trade-offs

**TSDF advantages:**
- Produces smooth, continuous surfaces via the zero-crossing
- Sub-voxel surface accuracy through interpolation
- Efficient update: only voxels near the surface are affected ($|\text{sdf}| \leq \mu$)
- Can extract high-quality triangle meshes with Marching Cubes

**Occupancy grid advantages:**
- Explicitly represents free space (critical for safe navigation)
- Probabilistic: naturally handles uncertainty and conflicting observations
- Simple to query: "Is this voxel safe to traverse?" → $p < 0.5$
- Well-suited for planning algorithms (A*, RRT, etc.)
- Robust to outliers through the log-odds formulation

In [ ]:
fig = plt.figure(figsize=(18, 7))

ax1 = fig.add_subplot(131, projection="3d")
if len(verts_world) > 0:
    stride = max(1, len(verts_world) // 4000)
    ax1.scatter(verts_world[::stride, 0], verts_world[::stride, 1],
               verts_world[::stride, 2], c=verts_world[::stride, 2],
               cmap="viridis", s=1, alpha=0.4)
ax1.set_title("TSDF → Mesh Vertices")
ax1.set_xlabel("X"); ax1.set_ylabel("Y"); ax1.set_zlabel("Z")

ax2 = fig.add_subplot(132, projection="3d")
if len(occ_pts) > 0:
    stride = max(1, len(occ_pts) // 4000)
    ax2.scatter(occ_pts[::stride, 0], occ_pts[::stride, 1],
               occ_pts[::stride, 2], c="red", s=2, alpha=0.5)
ax2.set_title("Occupancy Grid → Occupied")
ax2.set_xlabel("X"); ax2.set_ylabel("Y"); ax2.set_zlabel("Z")

ax3 = fig.add_subplot(133)
tsdf_mid = tsdf.dims[2] // 2
occ_mid = occ_grid.grid_size[2] // 2

tsdf_slice = tsdf.tsdf_vol[:, :, tsdf_mid].copy()
tsdf_slice[tsdf.weight_vol[:, :, tsdf_mid] == 0] = np.nan
occ_slice = prob_grid[:, :, occ_mid]

ax3.contour(tsdf_slice.T, levels=[0], colors="blue", linewidths=2)
ax3.contour(occ_slice.T, levels=[0.5], colors="red", linewidths=2,
            linestyles="dashed")
ax3.set_title("Mid-Z Contours: TSDF (blue) vs Occ (red)")
ax3.set_xlabel("X voxel")
ax3.set_ylabel("Y voxel")
ax3.set_aspect("equal")

plt.suptitle("TSDF vs Occupancy Grid — Side-by-Side Comparison", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
print("═" * 70)
print(f"{'Metric':<35} {'TSDF':>15} {'Occupancy':>15}")
print("─" * 70)
print(f"{'Grid resolution (m)':<35} {voxel_size:>15.3f} {occ_resolution:>15.3f}")
print(f"{'Grid dimensions':<35} {str(tuple(int(x) for x in tsdf.dims)):>15} {str(tuple(int(x) for x in occ_grid.grid_size)):>15}")
print(f"{'Total voxels':<35} {np.prod(tsdf.dims):>15,} {np.prod(occ_grid.grid_size):>15,}")

tsdf_observed = np.sum(tsdf.weight_vol > 0)
occ_known = np.sum(np.abs(occ_grid.grid) > 0.01)
print(f"{'Observed/known voxels':<35} {tsdf_observed:>15,} {occ_known:>15,}")

if len(verts_world) > 0:
    print(f"{'Extracted surface vertices':<35} {len(verts_world):>15,} {'N/A':>15}")
    print(f"{'Extracted triangles':<35} {len(faces):>15,} {'N/A':>15}")

print(f"{'Occupied voxels (p>0.6)':<35} {'N/A':>15} {len(occ_pts):>15,}")
print(f"{'Free voxels (p<0.3)':<35} {'N/A':>15} {len(free_pts):>15,}")

mem_tsdf = (tsdf.tsdf_vol.nbytes + tsdf.weight_vol.nbytes) / 1024
mem_occ = occ_grid.grid.nbytes / 1024
print(f"{'Memory (KB, grid only)':<35} {mem_tsdf:>15.1f} {mem_occ:>15.1f}")
print("═" * 70)

---
## 5. Efficient Map Storage (Discussion)

### 5.1 The Problem with Dense Grids

A dense 3D grid with side length $L$, resolution $r$, storing $b$ bytes per voxel requires:

$$\text{Memory} = \left(\frac{L}{r}\right)^3 \times b$$

| Volume | Resolution | Voxels | Memory (4 B/voxel) |
|--------|-----------|--------|-------------------|
| 10 m cube | 1 cm | $10^9$ | 4 GB |
| 10 m cube | 5 mm | $8 \times 10^9$ | 32 GB |
| 100 m cube | 5 cm | $8 \times 10^9$ | 32 GB |

The cubic scaling $O(n^3)$ makes dense grids impractical for large environments or
fine resolutions. Real-world spaces are **sparse** — most voxels are either entirely
free or unobserved. We need data structures that exploit this sparsity.

### 5.2 OctoMap (Octree-Based Occupancy Mapping)

**Key idea:** Represent the volume as an **octree**, where each node covers a cubic region
that is recursively subdivided into 8 children only when needed.

- Uniform regions (all free or all occupied) are stored as a single leaf → $O(1)$
- Only heterogeneous regions are subdivided
- Typical compression: 10–100× vs. dense grid for real-world environments
- Supports multi-resolution queries (coarse for planning, fine near surfaces)
- Uses the same log-odds probabilistic model as flat occupancy grids

**Memory complexity:** $O(n_{\text{surface}})$ rather than $O(L^3/r^3)$, where
$n_{\text{surface}}$ is proportional to the surface area of observed geometry.

**Reference:** Hornung et al., "OctoMap: An Efficient Probabilistic 3D Mapping Framework",
Autonomous Robots 2013.

### 5.3 Voxel Hashing

**Key idea:** Use a hash table to store only the voxels that have been observed. Each
voxel's 3D index $(i, j, k)$ is hashed to a table entry:

$$h(i, j, k) = (i \cdot p_1 \oplus j \cdot p_2 \oplus k \cdot p_3) \bmod N$$

where $p_1, p_2, p_3$ are large primes and $\oplus$ is XOR.

**Advantages:**
- $O(1)$ lookup, insert, and delete per voxel
- No wasted memory on empty regions
- Naturally supports unbounded environments (no predefined volume)
- GPU-friendly: can use parallel hash tables for real-time operation

**Reference:** Nießner et al., "Real-time 3D Reconstruction at Scale using Voxel Hashing",
ACM ToG 2013.

### 5.4 Spatial Hashing for Sparse Volumes

An extension of voxel hashing that organises voxels into **blocks** (e.g., $8^3$ voxels per
block). The hash table maps block indices to contiguous arrays of voxels:

$$\text{block\_idx} = \left\lfloor \frac{(i, j, k)}{B} \right\rfloor$$

where $B$ is the block side length. This improves cache coherency (nearby voxels are stored
together in memory) while maintaining the sparsity benefits of hashing.

### 5.5 Comparison of Storage Approaches

| Method | Memory | Lookup | Multi-res | GPU-friendly |
|--------|--------|--------|-----------|-------------|
| Dense grid | $O(L^3/r^3)$ | $O(1)$ | No | Yes |
| Octree | $O(n_{\text{surface}})$ | $O(\log n)$ | Yes | Limited |
| Voxel hash | $O(n_{\text{observed}})$ | $O(1)$ amortised | No | Yes |
| Spatial hash | $O(n_{\text{blocks}})$ | $O(1)$ amortised | Partial | Yes |

In [ ]:
print("Memory requirements for dense 3D grids:")
print("=" * 60)
print(f"{'Volume':>12} {'Resolution':>12} {'Voxels':>14} {'Memory':>12}")
print("-" * 60)

cases = [
    (2, 0.02), (5, 0.02), (10, 0.01), (10, 0.005),
    (50, 0.05), (100, 0.05), (100, 0.01),
]

for L, r in cases:
    n_voxels = (L / r) ** 3
    mem_bytes = n_voxels * 8
    if mem_bytes < 1e6:
        mem_str = f"{mem_bytes/1e3:.0f} KB"
    elif mem_bytes < 1e9:
        mem_str = f"{mem_bytes/1e6:.0f} MB"
    elif mem_bytes < 1e12:
        mem_str = f"{mem_bytes/1e9:.1f} GB"
    else:
        mem_str = f"{mem_bytes/1e12:.1f} TB"
    print(f"{L:>10} m  {r*100:>9.1f} cm  {n_voxels:>14.2e}  {mem_str:>12}")

### 5.6 Advanced Spatial Data Structures

All schemes below exploit the **sparsity of 3D space** — most voxels are empty.

| Approach | Key Idea | Memory | Use Case |
|:---|:---|:---|:---|
| **MrHash** (2025) | Multi-resolution hashing: fine voxels near surfaces, coarse elsewhere | 4× less than fixed-res | Real-time SLAM |
| **ASH** (Open3D) | GPU hash map with PyTorch tensor interface | Adaptive | Production TSDF (`o3d.t.geometry.VoxelBlockGrid`) |
| **Instant-NGP** (Müller et al., 2022) | Multi-resolution hash tables store **learned features** | ~16 MB room-scale | NeRF, neural SLAM |

**Instant-NGP hash encoding** — the key formula:

$$
\mathbf{f}(\mathbf{x}) = \bigoplus_{l=1}^{L} \text{interp}\bigl(\text{HashTable}_l(\mathbf{x})\bigr), \qquad
h(\mathbf{x}) = \left(\bigoplus_{d=1}^{3} x_d \cdot \pi_d\right) \bmod T
$$

where $\bigoplus$ is concatenation across $L$ resolution levels, $T$ is the table size, and $\pi_d$ are large primes. Unlike traditional hashing that stores explicit geometry, hash encoding stores a compressed learned representation decoded by a small MLP into colour and density — enabling **real-time NeRF** (NB 15).

---
## 6. KinectFusion (Newcombe et al. 2011)

### 6.1 The First Real-Time TSDF System

KinectFusion demonstrated that the Curless & Levoy TSDF integration scheme could run in
**real-time** (30 Hz) on consumer GPU hardware using the Microsoft Kinect depth sensor.
It was a landmark result that made dense 3D reconstruction accessible.

### 6.2 Pipeline Overview

The KinectFusion pipeline has three main stages:

**Stage 1: Depth Preprocessing**
- Bilateral filtering to reduce noise while preserving edges
- Back-projection to a vertex map $\mathbf{V}(u, v) = D(u, v) \cdot K^{-1} \begin{pmatrix} u \\ v \\ 1 \end{pmatrix}$
- Normal estimation via cross products of adjacent vertices: $\mathbf{N}(u,v) = (\mathbf{V}(u+1,v) - \mathbf{V}(u,v)) \times (\mathbf{V}(u,v+1) - \mathbf{V}(u,v))$

**Stage 2: Camera Tracking (ICP)**
- Track the camera pose frame-to-model using **point-to-plane ICP**
- The "model" is a synthetic depth/normal map rendered from the TSDF (see Stage 3)
- Minimises: $E(T) = \sum_i \left( \mathbf{n}_i^\top \left( T \mathbf{v}_i^{\text{live}} - \mathbf{v}_i^{\text{model}} \right) \right)^2$
- Linearised via small-angle approximation and solved with Gauss–Newton
- Uses a coarse-to-fine pyramid for robustness

**Stage 3: TSDF Integration + Raycasting**
- Integrate the new depth frame into the global TSDF volume (exactly as in Section 1)
- Raycast the TSDF to produce a synthetic depth/normal map for the next frame's ICP:
  - For each pixel, march a ray from the camera through the volume
  - Find the zero-crossing of the TSDF along the ray (sign change from + to −)
  - Interpolate to find the precise surface point and its normal

### 6.3 Key Innovations

1. **Frame-to-model tracking**: Instead of aligning each depth frame to the previous frame
   (drift accumulates), KinectFusion aligns each frame to a synthetic view of the
   **accumulated TSDF model**. This dramatically reduces drift.

2. **GPU parallelism**: Every stage is parallelised on the GPU:
   - TSDF integration: each voxel is independent (massively parallel)
   - Raycasting: each pixel ray is independent
   - ICP: reduction operations on GPU

3. **Dense, real-time reconstruction**: Prior systems either required offline processing
   (Curless & Levoy) or produced only sparse reconstructions (visual SLAM). KinectFusion
   showed that dense volumetric reconstruction could run in real-time.

### 6.4 Limitations

- **Fixed volume size**: The original system used a $512^3$ dense grid ($\approx$ 512 MB for
  TSDF + weights), limiting the reconstructable volume to about 3–4 m per side at 1 cm
  resolution.
- **No loop closure**: Drift in tracking was reduced but not eliminated.
- **No colour**: The original paper focused on geometry only.

Later systems (Voxel Hashing, InfiniTAM, BundleFusion) addressed these limitations.

### 6.5 Reference

Newcombe, R. A., Izadi, S., Hilliges, O., Molyneaux, D., Kim, D., Davison, A. J.,
Kohli, P., Shotton, J., Hodges, S., & Fitzgibbon, A. (2011).
*KinectFusion: Real-Time Dense Surface Mapping and Tracking.*
IEEE International Symposium on Mixed and Augmented Reality (ISMAR).

In [ ]:
print("KinectFusion — Performance Characteristics")
print("=" * 55)
print(f"{'Component':<30} {'Time (ms)':>12} {'Rate':>10}")
print("-" * 55)
print(f"{'Depth preprocessing':<30} {'1-2':>12} {'':>10}")
print(f"{'ICP tracking (3 levels)':<30} {'5-10':>12} {'':>10}")
print(f"{'TSDF integration':<30} {'2-3':>12} {'':>10}")
print(f"{'Raycasting':<30} {'5-8':>12} {'':>10}")
print("-" * 55)
print(f"{'Total per frame':<30} {'15-25':>12} {'30 Hz':>10}")
print()

for n in [256, 384, 512]:
    mem_mb = (n ** 3 * 8) / (1024 ** 2)
    side_m = n * 0.01
    print(f"  {n}³ volume @ 1cm: {side_m:.1f}m side, {mem_mb:.0f} MB")

### 6.5 Rolling (Moving) Volumes

#### The Problem

KinectFusion's original TSDF volume is **fixed in world space**. A $512^3$ grid at
1 cm resolution covers only $\sim$5 m per side. The camera **cannot leave this
volume** — once it moves beyond the grid boundary, tracking and integration fail.

#### Solution: Shift the Volume Origin

As the camera approaches the volume boundary, **translate the volume** by one
block (e.g., $8 \times \text{voxel\_size}$) in the direction of camera motion:

1. Detect when camera position is within a margin of the volume boundary
2. Shift the volume origin by one block along the relevant axis
3. **Discard** voxels that fall outside the new bounds (they have already been
   meshed or transmitted)
4. **Initialise** newly exposed voxels as unknown ($\text{tsdf} = 1$, $w = 0$)

#### Circular Buffer Implementation

The key trick is **circular (modular) indexing** — no data is physically
copied when the volume shifts. Instead, the mapping from world voxel index
to storage index wraps around:

$$
\text{storage\_idx}(i) = i \bmod N
$$

where $N$ is the grid dimension. When the volume shifts by $\Delta$ voxels,
the "oldest" slice of voxels is overwritten with unknown values. This gives
$O(N^2)$ work per shift (resetting one slice) instead of $O(N^3)$ for copying
the entire volume.

#### From KinectFusion to InfiniTAM

| System | Volume | Coverage |
|:---|:---|:---|
| KinectFusion (2011) | Fixed $512^3$ grid | ~5 m cube |
| Kintinuous (2012) | Rolling volume + mesh streaming | Unbounded (room-scale) |
| InfiniTAM (2015) | Voxel hashing (no fixed grid) | Unbounded (building-scale) |

The moving volume was a pragmatic intermediate step. Modern systems (InfiniTAM,
voxel hashing, Section 5.3) eliminate the fixed grid entirely, but the circular
buffer idea remains useful in memory-constrained embedded systems (drones, AR
glasses) where a hash table's overhead is too high.

Rolling volumes are particularly important for drones, which continuously explore new areas but have limited onboard memory — the circular buffer lets them maintain a local map around the current position without ever running out of memory.

### 6.6 Point Cloud Compression

A dense depth frame (640×480) produces ~3.6 MB of XYZ data. At 30 Hz that is ~100 MB/s — far exceeding multi-robot bandwidth. Compression is essential for collaborative mapping.

| Scenario | Bandwidth | Compression needed |
|:---|:---|:---|
| Single robot, onboard | N/A (local) | None |
| Multi-robot, WiFi mesh | 5–50 Mbps shared | 10–50× |
| Robot → cloud, cellular | 1–10 Mbps | 50–100× |

**Classical**: *Draco* (Google) — quantisation + predictive coding, 2–10× reduction, widely supported (glTF, Open3D, ROS). **Learned**: *OctGLP-Net* — octree + neural entropy model, outperforms Draco/G-PCC at low bitrates.

## 7. BEV Perception & Occupancy Networks (Production Systems)

Production autonomous driving uses **Bird's-Eye View (BEV)** pipelines that lift multi-camera images into a **3D occupancy grid** viewed from above:

| System | View Transform | Key Idea |
|:---|:---|:---|
| **LSS** (Philion & Fidler, 2020) | Explicit depth → lift + splat | Predicts per-pixel depth distribution |
| **BEVFormer** (Li et al., 2022) | Spatial cross-attention | Learnable BEV queries, no explicit depth |
| **BEVDet** (Huang et al., 2022) | LSS-style, efficient | BEVDet4D adds temporal fusion |
| **Tesla Occupancy Net** | Spatial attention transformer | 200×200×16 voxel grid, predicts occupancy + velocity |

**Key insight**: predict *what is in each voxel* (occupied/free) rather than *what objects are in the scene* — naturally handles novel obstacles.

**4D occupancy world models** (OccWorld, SparseWorld) extend this by forecasting future scene evolution — enabling a drone to predict where obstacles will be in 2 seconds.

**LiDAR-camera fusion** (R3LIVE++, FAST-LIVO2): most production systems fuse LiDAR + camera + IMU. These require ROS and specialised hardware, but architecturally explain why visual-only systems trade accuracy for simplicity.

---
## 7. Exercises

### Exercise 1: Implement TSDF from Scratch (Guided)

Complete the TSDF integration for a **single voxel**. Given:
- Voxel world position $\mathbf{v}$
- Camera intrinsics $K$, pose $T_{\text{cam}\to\text{world}}$
- Depth image $D$

Implement each step of the algorithm.

In [ ]:
def tsdf_integrate_single_voxel(
    voxel_world, depth, K, T_cam_to_world, trunc_dist,
    tsdf_old, weight_old, weight_new=1.0
):
    """
    Exercise 1: Implement TSDF integration for a single voxel.

    Parameters
    ----------
    voxel_world : (3,) array — voxel position in world coordinates
    depth       : (H, W) array — depth image
    K           : (3, 3) — camera intrinsic matrix
    T_cam_to_world : (4, 4) — camera-to-world SE(3)
    trunc_dist  : float — truncation distance
    tsdf_old    : float — current TSDF value at this voxel
    weight_old  : float — current weight
    weight_new  : float — weight of this new observation

    Returns
    -------
    tsdf_new : float — updated TSDF value
    weight   : float — updated weight
    valid    : bool  — whether the voxel was updated
    """
    H, W = depth.shape

    # Step 1: Transform voxel to camera frame
    # YOUR CODE: T_world_to_cam = ...
    # YOUR CODE: p_cam = T_world_to_cam @ [voxel_world; 1]

    # Step 2: Project to pixel coordinates
    # YOUR CODE: u = fx * (Xc / Zc) + cx
    # YOUR CODE: v = fy * (Yc / Zc) + cy

    # Step 3: Bounds check
    # YOUR CODE: check Zc > 0, 0 <= u < W, 0 <= v < H

    # Step 4: Compute SDF
    # YOUR CODE: d_obs = depth[round(v), round(u)]
    # YOUR CODE: sdf = d_obs - Zc

    # Step 5: Truncate
    # YOUR CODE: tsdf_meas = clamp(sdf / trunc_dist, -1, 1)

    # Step 6: Weighted average
    # YOUR CODE: tsdf = (w_old * tsdf_old + w_new * tsdf_meas) / (w_old + w_new)

    # Placeholder return — replace with your implementation
    return tsdf_old, weight_old, False


test_voxel = np.array([0.0, 0.0, 0.0])
t_new, w_new, valid = tsdf_integrate_single_voxel(
    test_voxel, depth_frames[0], K, poses[0],
    trunc_dist=0.12, tsdf_old=1.0, weight_old=0.0
)
print(f"Voxel {test_voxel}: tsdf={t_new:.4f}, weight={w_new:.1f}, valid={valid}")
print("(Expected: valid=True with tsdf ∈ [-1, 1] once implemented)")

### Exercise 2: Integrate 10 Depth Frames from a Synthetic Moving Camera

Using the `TSDFVolumeFromScratch` class, create a TSDF volume and integrate at least 10
of the pre-generated depth frames. Track how the number of observed voxels grows.

**Tasks:**
1. Create a TSDF volume with appropriate bounds and resolution
2. Loop through the first 10 frames and integrate each one
3. After each integration, count the number of voxels with weight > 0
4. Plot the growth of observed voxels over frames

In [ ]:
tsdf_ex2 = TSDFVolumeFromScratch(vol_bounds, voxel_size, trunc_dist)

observed_counts = []
for i in range(min(10, n_frames)):
    tsdf_ex2.integrate(depth_frames[i], K, poses[i])
    n_observed = int(np.sum(tsdf_ex2.weight_vol > 0))
    observed_counts.append(n_observed)
    print(f"  Frame {i+1:2d}: {n_observed:>6d} observed voxels "
          f"({n_observed / tsdf_ex2.weight_vol.size * 100:.1f}%)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(observed_counts) + 1), observed_counts, 'bo-', linewidth=2)
ax.set_xlabel("Frames integrated")
ax.set_ylabel("Observed voxels (weight > 0)")
ax.set_title("TSDF Coverage vs. Number of Integrated Frames")
ax.grid(True, alpha=0.3)
ax.set_xticks(range(1, len(observed_counts) + 1))
plt.tight_layout()
plt.show()

print(f"\nCoverage doubles from {observed_counts[0]} to {observed_counts[-1]} voxels "
      f"— each viewpoint reveals previously unseen surfaces.")

### Exercise 3: Extract Mesh Using Marching Cubes, Visualise

After integrating the depth frames (Exercise 2), use `skimage.measure.marching_cubes`
to extract the mesh and visualise it.

**Tasks:**
1. Prepare the TSDF volume (set unobserved voxels to +1)
2. Run marching cubes at level 0
3. Convert vertex coordinates from voxel to world frame
4. Visualise the mesh using `Poly3DCollection`

In [ ]:
from skimage.measure import marching_cubes

tsdf_vol = tsdf_ex2.tsdf_vol.copy()
tsdf_vol[tsdf_ex2.weight_vol == 0] = 1.0

try:
    verts, faces, normals, _ = marching_cubes(tsdf_vol, level=0.0)
    verts_world = verts * tsdf_ex2.voxel_size + tsdf_ex2.origin

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_trisurf(
        verts_world[:, 0], verts_world[:, 1], verts_world[:, 2],
        triangles=faces, cmap='viridis', alpha=0.6, edgecolor='none')

    cam_positions = np.array([p[:3, 3] for p in poses[:10]])
    ax.scatter(cam_positions[:, 0], cam_positions[:, 1], cam_positions[:, 2],
               c='red', s=40, marker='^', label='Cameras')

    ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]'); ax.set_zlabel('Z [m]')
    ax.set_title(f'Marching Cubes Mesh ({len(verts)} vertices, {len(faces)} faces)')
    ax.legend()
    ax.view_init(elev=25, azim=-60)
    plt.tight_layout()
    plt.show()
except (RuntimeError, ValueError) as e:
    print(f"Marching cubes failed: {e} (TSDF may not contain a zero-crossing)")

### Exercise 4: Implement Occupancy Grid from Scratch

Using the `OccupancyGridFromScratch` class as a guide, implement your own occupancy grid
with the following modifications:

**Tasks:**
1. Create an occupancy grid for the same volume
2. Integrate all 12 depth frames
3. Visualise the log-odds distribution as a histogram
4. Show a 2D slice of the probability grid

In [ ]:
occ_ex4 = OccupancyGridFromScratch(
    bounds=vol_bounds, resolution=0.08,
    log_odds_occ=2.0, log_odds_free=-0.5, log_odds_max=5.0)

for i in range(min(10, n_frames)):
    occ_ex4.update(depth_frames[i], K, poses[i])

prob = occ_ex4.to_probability()
mid_z = occ_ex4.grid_size[2] // 2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(occ_ex4.grid[:, :, mid_z].T, origin='lower', cmap='RdBu_r',
                       vmin=-5, vmax=5)
axes[0].set_title(f'Log-Odds (z-slice {mid_z})')
axes[0].set_xlabel('X index'); axes[0].set_ylabel('Y index')
plt.colorbar(im0, ax=axes[0], label='Log-odds l(v)')

im1 = axes[1].imshow(prob[:, :, mid_z].T, origin='lower', cmap='hot',
                       vmin=0, vmax=1)
axes[1].set_title(f'Occupancy Probability (z-slice {mid_z})')
axes[1].set_xlabel('X index'); axes[1].set_ylabel('Y index')
plt.colorbar(im1, ax=axes[1], label='p(occupied)')

plt.suptitle('Occupancy Grid: Log-Odds vs Probability', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

occ_pts = occ_ex4.get_occupied_points()
free_pts = occ_ex4.get_free_points()
print(f"Occupied voxels: {len(occ_pts)}, Free voxels: {len(free_pts)}")

### Exercise 5: Compare TSDF Surface vs. Occupancy Grid on the Same Data

Using the results from Exercises 2–4, create a combined visualisation:

**Tasks:**
1. Extract the TSDF surface (marching cubes vertices)
2. Extract occupied voxels from the occupancy grid
3. Plot both in the same 3D figure with different colours
4. Create a 2D overlay showing both contours on the same slice
5. Discuss: Where do they agree? Where do they differ? Why?

In [ ]:
fig = plt.figure(figsize=(16, 7))

ax1 = fig.add_subplot(121, projection='3d')
if 'verts_world' in dir() and len(verts_world) > 0:
    sub = np.random.choice(len(verts_world), min(3000, len(verts_world)), replace=False)
    ax1.scatter(verts_world[sub, 0], verts_world[sub, 1], verts_world[sub, 2],
                s=1, c=verts_world[sub, 2], cmap='viridis', alpha=0.6)
ax1.set_title(f'TSDF Surface ({len(verts_world) if "verts_world" in dir() else 0} vertices)')
ax1.set_xlabel('X [m]'); ax1.set_ylabel('Y [m]'); ax1.set_zlabel('Z [m]')
ax1.view_init(elev=25, azim=-60)

ax2 = fig.add_subplot(122, projection='3d')
occ_pts = occ_ex4.get_occupied_points()
if len(occ_pts) > 0:
    sub_o = np.random.choice(len(occ_pts), min(3000, len(occ_pts)), replace=False)
    ax2.scatter(occ_pts[sub_o, 0], occ_pts[sub_o, 1], occ_pts[sub_o, 2],
                s=2, c=occ_pts[sub_o, 2], cmap='hot', alpha=0.5)
ax2.set_title(f'Occupancy Grid ({len(occ_pts)} occupied)')
ax2.set_xlabel('X [m]'); ax2.set_ylabel('Y [m]'); ax2.set_zlabel('Z [m]')
ax2.view_init(elev=25, azim=-60)

plt.suptitle('TSDF (continuous surface) vs Occupancy (binary free/occupied)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("TSDF: sub-voxel precision, ideal for meshing and visualisation")
print("Occupancy: binary free/occupied, ideal for A*/RRT* path planning")
print("TSDF surfaces are smoother; occupancy captures full volumetric state")

---

## Summary

In this notebook we derived and implemented two foundational volumetric mapping representations:

1. **TSDF** — stores signed distance to the nearest surface, fused via weighted running
   average (Curless & Levoy 1996). Surfaces are extracted as the zero-level set using
   **Marching Cubes** (Lorensen & Cline 1987).

2. **Occupancy Grid** — stores occupancy probability in log-odds form, updated via Bayesian
   inference with an inverse sensor model. Ray casting uses **Bresenham's 3D line algorithm**
   to mark voxels as free or occupied.

Key takeaways:
- **TSDF** is optimal for **surface reconstruction** — smooth meshes, sub-voxel accuracy
- **Occupancy grids** are optimal for **navigation** — explicit free space, probabilistic reasoning
- Dense grids scale as $O(n^3)$ and don't work for large environments; use **OctoMap** or
  **voxel hashing** instead
- **KinectFusion** (Newcombe et al. 2011) brought real-time TSDF to consumer hardware

### Next Steps
- Notebook 13: IMU sensor fusion and visual-inertial odometry
- Notebook 15: Neural 3D reconstruction (NeRF, 3DGS) — learned alternatives to TSDF